# OTT Streaming Content Trend Analysis
**Netflix catalog snapshot | Python / Pandas / Matplotlib**

The supplied dataset is Netflix-only. This notebook performs verified Netflix analysis and adds a platform field for future multi-platform integration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns',30)

In [ ]:
df=pd.read_csv('../data/raw/netflix_titles.csv')
print(df.shape)
df.head()

## Data quality check
Check missing values, duplicate records and key uniqueness before analysis.

In [ ]:
print(df.isna().sum())
print('Duplicate rows:',df.duplicated().sum())
print('Duplicate show_id:',df.show_id.duplicated().sum())

## Cleaning and enrichment
Convert dates, extract duration, and prepare platform, genre and country fields.

In [ ]:
df['date_added']=pd.to_datetime(df['date_added'],errors='coerce')
df['added_year']=df['date_added'].dt.year
d=df['duration'].str.extract(r'(?P<duration_value>\d+)\s*(?P<duration_unit>.*)')
df['duration_value']=pd.to_numeric(d['duration_value'],errors='coerce')
df['duration_unit']=d['duration_unit'].str.strip()
df['platform']='Netflix'

## Content mix

In [ ]:
type_summary=df['type'].value_counts()
print(type_summary)
type_summary.plot.bar(title='Movies vs TV Shows'); plt.ylabel('Titles'); plt.show()

## Genre analysis

In [ ]:
g=df[['show_id','listed_in']].dropna().copy(); g['genre']=g['listed_in'].str.split(','); g=g.explode('genre'); g['genre']=g['genre'].str.strip()
print(g.groupby('genre')['show_id'].nunique().sort_values(ascending=False).head(10))

## Titles added over time

In [ ]:
added=df.dropna(subset=['added_year']).groupby('added_year')['show_id'].nunique()
added.plot(marker='o',title='Titles Added to Netflix by Year'); plt.ylabel('Titles'); plt.show()

## Country and rating analysis

In [ ]:
c=df[['show_id','country']].dropna().copy(); c['country_name']=c['country'].str.split(','); c=c.explode('country_name'); c['country_name']=c['country_name'].str.strip()
print('Top countries:\n',c.groupby('country_name')['show_id'].nunique().sort_values(ascending=False).head(10))
print('\nRatings:\n',df['rating'].fillna('Not Rated').value_counts().head(10))

## Business storytelling
The findings are descriptive and reflect this historical catalog snapshot. They do not measure views, revenue, subscribers or current market share.

For a true Netflix/Prime/Hotstar comparison, standardize equivalent datasets to the same schema and concatenate them using the platform field.